# Test - Automated Test Generation Agent

In this exercise, we build an agent that takes an existing Python repository, measures its current test coverage, and writes new `pytest` tests to cover the parts of the codebase that are currently untested.

The agent runs locally on a checkout of the repo. Once the generated tests pass locally, we optionally open a Pull Request with the new tests via the official GitHub MCP server.

## Environment Variables & Imports

Before using this notebook, please ensure you have obtained an API Key from your LLM backend and update the `.env` file to include it as follows:

```bash
GOOGLE_API_KEY=<copy API key here>
OPENAI_API_KEY=<copy API key here>
ANTRHOPIC_API_KEY=<copy API key here>
LLAMA_CPP_API_KEY=<copy API key here>
```


For the optional PR flow, add a `GITHUB_PERSONAL_ACCESS_TOKEN` to `.env` and make sure the `code_folder` is a git clone of the target repository with the proper remote configured.

In [ ]:
import initialize_notebook # noqa

In [ ]:
import json
import os
import pathlib

import jinja2

from hslu.dlm03.common import backend as backend_lib
from hslu.dlm03.common import chat as chat_lib
from hslu.dlm03.common import presets
from hslu.dlm03.common.displays import ipython_display
from hslu.dlm03.common import tools as tools_lib
from hslu.dlm03.util import file as file_lib

## Parameters

In [ ]:
backend_name = "Gemma 4 26B"
backend_config = presets.CONFIGS_BY_NAME[backend_name]
backend = backend_lib.LLM.from_config(backend_config)

In [ ]:
# Local checkout of the target repository.
code_folder = pathlib.Path("/Users/vincent/Development/Bank-Account/")
# Directory where new tests will be written (relative to code_folder, or absolute).
test_folder = code_folder / "tests"

# Optional: if you want the agent to open a PR with the new tests, configure these.
github_mcp_url = "https://api.githubcopilot.com/mcp/"
github_token = os.environ.get("GITHUB_PAT")
repo_owner = "VincentCoriou"
repo_name = "Bank-Account"
base_branch = "main"
open_pull_request = False  # Flip to True once the local flow works.

# MCP Server

We expose read-only filesystem tools scoped to the repository, a `write_test_file` tool scoped to the test folder, and `run_coverage` / `run_pytest` for the agent to observe the effect of its changes.

In [ ]:
import subprocess

from mcp.server import FastMCP

SERVER = FastMCP()

CODE_FOLDER = code_folder.resolve()
TEST_FOLDER = test_folder.resolve()


@SERVER.tool()
def list_files(glob: str = "**/*") -> list[str]:
    """List the files in `path` that match the given glob expression, returning absolute paths.

    Prefer narrow globs (e.g. `*.py`, `requirements*.txt`) to avoid very large listings.
    """
    root = CODE_FOLDER
    files = [p for p in root.glob(glob) if p.is_file()]
    if not files:
        return "No files found."
    return [str(p) for p in root.glob(glob) if p.is_file() and ".git" not in str(p)]


@SERVER.tool()
def read_file(path: str) -> file_lib.File | str:
    """Read a file. `path` can be absolute or relative to the repo root."""
    candidate = pathlib.Path(path)
    file_path = candidate if candidate.is_absolute() else (CODE_FOLDER / candidate)
    if not file_path.exists():
        return "File not found."
    if file_path.is_dir():
        return "Given path is a directory."
    return file_lib.File.read_from(file_path)


@SERVER.tool()
def write_test_file(filename: str, content: str) -> str:
    """Write a pytest test file into the configured test folder."""
    if ".." in filename or pathlib.Path(filename).is_absolute():
        return "filename must be a simple relative path inside the test folder."
    target = TEST_FOLDER / filename
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content)
    return f"Wrote {target}"


@SERVER.tool()
def run_coverage(pytest_args: str = "") -> str:
    """Run `pytest` with coverage and return a summary of uncovered lines.

    Focuses on Python files directly under CODE_FOLDER (not test files or venv).
    """
    cmd = [
        "python", "-m", "pytest",
        f"--cov={CODE_FOLDER}",
        "--cov-report=term-missing:skip-covered",
        "--cov-report=json:coverage.json",
        "-q",
    ]
    if pytest_args.strip():
        cmd.extend(pytest_args.split())
    try:
        result = subprocess.run(
            cmd, cwd=CODE_FOLDER, capture_output=True, text=True, timeout=300,
        )
    except FileNotFoundError:
        return "pytest / coverage not installed in the active env."
    except subprocess.TimeoutExpired:
        return "Coverage run timed out after 5 minutes."
    output = (result.stdout or "") + "\n" + (result.stderr or "")
    return f"exit_code={result.returncode}\n{output[-5000:]}"


@SERVER.tool()
def run_pytest(test_path: str = "") -> str:
    """Run `pytest` (optionally against a specific path) and return the output."""
    cmd = ["python", "-m", "pytest", "-q"]
    if test_path.strip():
        cmd.append(test_path)
    try:
        result = subprocess.run(
            cmd, cwd=CODE_FOLDER, capture_output=True, text=True, timeout=300,
        )
    except FileNotFoundError:
        return "pytest not installed in the active env."
    except subprocess.TimeoutExpired:
        return "pytest run timed out after 5 minutes."
    output = (result.stdout or "") + "\n" + (result.stderr or "")
    return f"exit_code={result.returncode}\n{output[-5000:]}"

In [ ]:
import threading

import uvicorn

PORT = 5000
HOST = "localhost"

RUN_ARGS = {
    "app": SERVER.streamable_http_app,
    "port": PORT,
    "host": HOST,
    "log_level": "warning",
}

MCP_THREAD = threading.Thread(target=uvicorn.run, kwargs=RUN_ARGS, daemon=True)
MCP_THREAD.start()

MCP_URL = f"http://{HOST}:{PORT}/mcp"

# Agentic Harness

In [ ]:
import re
def remove_thoughts(text: str) -> str:
    # Remove well-formed <thought>...</thought> blocks (multiline safe)
    cleaned = re.sub(r"<thought>.*?</thought>", "", text, flags=re.DOTALL | re.IGNORECASE)

    # Optionally handle unclosed <thought> tags (strip until end)
    cleaned = re.sub(r"<thought>.*$", "", cleaned, flags=re.DOTALL | re.IGNORECASE)

    return cleaned.strip()

In [ ]:
system_instructions_template_string = \
"""You are an expert Python engineer specialized in writing tests.

Given a local Python repository and its current pytest suite, your goal is to increase test
coverage by writing new `pytest` tests for code paths that are not currently covered.

Follow this workflow:
1. Run `run_coverage` to see the current coverage report and identify uncovered functions /
   branches.
2. Use `list_files` + `read_file` to read the relevant source files AND any existing tests
   (match the project's existing style, fixtures, and import conventions).
3. Pick one cohesive unit to cover (a single module or class is a good scope). Design tests
   that:
   - target the uncovered lines / branches,
   - exercise both happy paths and edge cases,
   - are deterministic and self-contained (no network, no sleeps),
   - follow pytest idioms (use `parametrize` and fixtures where natural).
4. Write the tests with `write_test_file` and run `run_pytest` against the new file. If
   tests fail, inspect the output, fix the tests (do not modify source code), and retry.
5. Once the new tests pass, run `run_coverage` again to confirm coverage increased, then
   stop and summarize what you added.
"""
system_instructions_template = jinja2.Template(system_instructions_template_string, undefined=jinja2.StrictUndefined)

In [ ]:
user_message_template_string = \
"""Please add tests to improve coverage for the repository at {{ code_folder }}.
Write new test files into {{ test_folder }}."""
user_message_template = jinja2.Template(user_message_template_string, undefined=jinja2.StrictUndefined)

In [ ]:
system_instructions = system_instructions_template.render()
user_message = user_message_template.render(code_folder=code_folder, test_folder=test_folder)

chat_display = ipython_display.IPythonChatDisplay()
chat_display.show()
chat = chat_lib.Chat(
    messages=[
        {"role": "system", "content": system_instructions},
        {"role": "user", "content": user_message},
    ],
    observers=[chat_display],
)
async with tools_lib.mcp_session(MCP_URL) as session:
    mcp_tools = await session.list_tools()
    tools = [tools_lib.tool_from_mcp(tool) for tool in mcp_tools.tools]
    done = False
    while not done:
        response = backend.generate(chat, tools=tools)
        done = True
        message = response.choices[0].message
        if message.content is not None:
            message.content = remove_thoughts(message.content)
        chat.append(message)
        for tool_call in message.tool_calls or ():
            done = False
            tool_name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)
            tool_call_result = await session.call_tool(tool_name, arguments)
            for content in tool_call_result.content:
                tool_call_result_content = tools_lib.tool_call_result_from_mcp(
                    tool_call.id,
                    content,
                )
                chat.append(tool_call_result_content)